# Semantics Continued Pretraining (DAPT) & Corpus Building Pipeline

This notebook runs the Semantics Layer pipeline directly in Google Colab.

### Assumptions:
- Your project folder (containing `data`, `evals`, `lib`, and `models`) is stored in your Google Drive.
- You have a GPU runtime enabled in Colab (highly recommended for CPT step `s2`).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Directory Path and Change Working Directory

Set `GOOGLE_DRIVE_PATH` to the location of the project folder in your Google Drive.

In [ ]:
import os
import sys

# Adjust this path to the location of your project folder in Google Drive
GOOGLE_DRIVE_PATH = "/content/drive/MyDrive/Semantics"

if os.path.exists(GOOGLE_DRIVE_PATH):
    os.chdir(GOOGLE_DRIVE_PATH)
    sys.path.append(GOOGLE_DRIVE_PATH)
    print(f"Successfully changed working directory to: {os.getcwd()}")
else:
    print(f"Error: Google Drive path '{GOOGLE_DRIVE_PATH}' does not exist. Please check your path.")

## 3. Install Dependencies

Install the required libraries on the Colab runtime environment.

In [ ]:
# Install dependencies required by Docling and Continued Pretraining (CPT/DAPT)
!pip install -q docling python-dotenv transformers accelerate torch boto3 tiktoken hf-xet pydrive2

## 4. Load Environment Variables & Handle Path Overrides

Loads settings from `.env`. If running in Colab (Linux), it dynamically overrides absolute Windows paths in `.env` with platform-neutral relative paths.

In [ ]:
import os
from dotenv import load_dotenv
from lib.utils import PipelineConfig

# Load configurations from .env file
load_dotenv()

# Override absolute Windows paths to relative paths if on Colab/Linux
if os.name != 'nt':
    print("Detected Linux/Colab environment. Overriding absolute Windows paths with cross-platform relative paths...")
    os.environ["LOCAL_DIRECTORY_PATH"] = "./data/raw"
    os.environ["OUTPUT_PATH"] = "./data/dapt/domain_dapt_corpus.jsonl"
    os.environ["PROBE_QA_PATH"] = "./evals/dapt/probe_qa.jsonl"

# Instantiate and validate pipeline config
cfg = PipelineConfig()
cfg.validate()
cfg.ensure_dirs()

print("\nCurrent Environment Configurations:")
print(f"STORAGE_TARGET: {cfg.build.storage_target}")
print(f"LOCAL_DIRECTORY_PATH: {cfg.build.local_directory_path}")
print(f"OUTPUT_PATH: {cfg.build.output_path}")
print(f"PROBE_QA_PATH: {cfg.data.qa_probe_path}")
print(f"BASE_MODEL_NAME: {cfg.model.base_model_name}")
print(f"MODEL_DTYPE: {cfg.model.model_dtype}")
print(f"MAX_SEQ_LEN: {cfg.model.max_seq_len}")
print(f"TOTAL_CORPUS_TOKENS: {cfg.corpus.total_corpus_tokens}")
print(f"MAX_CORPUS_PASSES: {cfg.corpus.max_corpus_passes}")
print(f"EVAL_INTERVAL_TOKENS: {cfg.corpus.eval_interval_tokens}")
print(f"DAPT_LR: {cfg.optimizer.learning_rate}")
print(f"DAPT_BATCH_SIZE: {cfg.optimizer.train_batch_size}")
print(f"CHUNK_SIZE: {cfg.build.chunk_size}")
print(f"WANDB_ENABLED: {cfg.wandb.enabled}")
print(f"WANDB_PROJECT: {cfg.wandb.project}")
print(f"WANDB_RUN_NAME: {cfg.wandb.run_name}")

In [ ]:
import time
def calculate_time(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"Function '{func.__name__}' took {end - start:.6f} seconds to run.")
        return result
    return wrapper


## 5. Step 1: Corpus Construction (s1)

Extracts and parses raw PDF documents from `data/raw` to build the pretraining corpus.

In [ ]:
from lib.s1_build_corpus import run_corpus_builder

@calculate_time
def s1(cfg):
    print("Starting Step 1 (Corpus Construction)...")
    run_corpus_builder(
        output_path=str(cfg.build.output_path),
        storage_target=cfg.build.storage_target,
        local_directory_path=cfg.build.local_directory_path,
        aws_bucket_name=cfg.build.aws_bucket_name,
        aws_prefix=cfg.build.aws_prefix,
        gdrive_folder_id=cfg.build.gdrive_folder_id,
        available_gpus=cfg.build.available_gpus,
        workers_per_gpu=cfg.build.workers_per_gpu,
        chunk_size=cfg.build.chunk_size,
    )
    print("Step 1 completed successfully!")

s1(cfg)

## 6. Step 2: Continued Pretraining (DAPT) & Evaluation (s2)

Performs domain adaptive pretraining (DAPT) on the base model using the chunked training blocks. Evaluates perplexity and QA accuracy before and after CPT.

In [ ]:
from lib.s2_dapt import run_dapt_pipeline

@calculate_time
def s2(cfg):
    print("Starting Step 2 (DAPT continued pretraining & evaluation)...")
    run_dapt_pipeline(
        model_name=cfg.model.base_model_name,
        corpus_path=str(cfg.build.output_path),
        probe_qa_path=str(cfg.data.qa_probe_path),
        epochs=cfg.corpus.max_corpus_passes,
        lr=cfg.optimizer.learning_rate,
        batch_size=cfg.optimizer.train_batch_size,
        output_dir=str(cfg.storage.checkpoint_dir),
    )
    print("Step 2 completed successfully!")

s2(cfg)